# SQLite 준비와 함수표

SQLite를 노트북(VS Code·Jupyter)에서 쓰기 위한 **설치·실행 안내**와, 이 대화에서 다룬 기능을 중심으로 한 **SQLite 함수표**입니다.
함수표의 *결과* 칸은 SQLite 3.45.1에서 직접 실행한 값입니다.

## 1. 설치와 실행

**SQLite는 따로 설치하지 않습니다.** 파이썬에 `sqlite3` 모듈이 내장되어 있습니다. 결과를 표로 보려고 `pandas`만 필요합니다.

| 순서 | 할 일 |
|---|---|
| 1 | 파이썬 설치 확인: 터미널에서 `python --version` |
| 2 | VS Code 확장 설치: **Python**, **Jupyter** (Microsoft) |
| 3 | 터미널에서 `pip install pandas` |
| 4 | `.ipynb` 파일을 열고 우측 상단 **커널 선택**에서 위 파이썬을 고름 |
| 5 | 셀을 위에서부터 `Shift+Enter`로 실행 |

**자주 막히는 곳**
- `ModuleNotFoundError: pandas` → `pip`을 실행한 파이썬과 노트북 커널이 서로 다른 파이썬입니다. 커널을 바꾸세요.
- 여러 문장을 한 번에 실행하려면 `con.execute`가 아니라 `con.executescript`를 쓰거나, 아래의 `sql()` 도우미를 쓰세요. (`execute`는 문장 하나만 받습니다.)
- 이 노트북의 DB는 **메모리 DB**(`":memory:"`)라서 커널을 끄면 사라집니다. 파일로 남기려면 `sqlite3.connect("practice.db")`로 바꾸세요.

In [1]:
import re, sqlite3
import pandas as pd
try:
    from IPython.display import display
except ImportError:          # 일반 파이썬에서 실행할 때
    display = print

con = sqlite3.connect(":memory:")   # 메모리 DB: 노트북(커널)을 끄면 사라집니다

def sql(text):
    """SQL을 실행합니다. SELECT류는 표로 보여주고, CREATE/INSERT 등은 실행만 합니다."""
    stmts, buf = [], ""
    for line in text.splitlines(keepends=True):
        buf += line
        if sqlite3.complete_statement(buf):      # ; 로 끝나는 문장 단위로 분리
            stmts.append(buf.strip()); buf = ""
    if buf.strip():
        stmts.append(buf.strip())
    ran = shown = 0
    for st in stmts:
        body = re.sub(r"--[^\n]*|/\*.*?\*/", "", st, flags=re.S).strip().rstrip(";").strip()
        if not body:
            continue                              # 주석만 있는 조각은 건너뜀
        ran += 1
        if body.split(None, 1)[0].upper() in ("SELECT", "WITH", "PRAGMA", "VALUES"):
            display(pd.read_sql_query(st, con)); shown += 1
        else:
            con.execute(st)
    con.commit()
    if ran == 0:
        print("(아직 작성한 SQL이 없습니다. 주석 아래에 쿼리를 작성한 뒤 다시 실행하세요.)")
    elif shown == 0:
        print("실행 완료")

print("SQLite 버전:", sqlite3.sqlite_version)

SQLite 버전: 3.53.4


## 2. 내 환경에서 되는 기능 점검
버전에 따라 일부 기능이 없을 수 있어서, 실제로 실행해 보고 OK/미지원을 표시합니다.

In [2]:
CHECKS = [
    ("윈도우 함수 (RANK, LAG …)", "SELECT RANK() OVER (ORDER BY 1)", "3.25+"),
    ("NULLS FIRST / LAST", "SELECT 1 ORDER BY 1 NULLS LAST", "3.30+"),
    ("FILTER (WHERE …)", "SELECT COUNT(*) FILTER (WHERE 1)", "3.30+"),
    ("IIF()", "SELECT IIF(1, 'a', 'b')", "3.32+"),
    ("RIGHT / FULL JOIN", "SELECT 1 FROM (SELECT 1 a) x FULL JOIN (SELECT 1 a) y ON x.a = y.a", "3.39+"),
    ("JSON 함수 json_extract", "SELECT json_extract('{\"a\": 1}', '$.a')", "3.38+ 기본 포함"),
    ("JSON ->> 연산자", "SELECT '{\"a\": 1}' ->> '$.a'", "3.38+"),
    ("CONCAT()", "SELECT CONCAT('a', 'b')", "3.44+"),
    ("STRING_AGG()", "SELECT STRING_AGG('a', ',')", "3.44+"),
    ("수학 함수 floor/sqrt/power", "SELECT floor(2.5), sqrt(4), power(2, 3)", "빌드에 따라 다름"),
]
rows = []
for name, q, need in CHECKS:
    try:
        con.execute(q).fetchall(); ok = "OK"
    except sqlite3.Error:
        ok = "미지원"
    rows.append((name, ok, need))
pd.DataFrame(rows, columns=["기능", "내 환경", "필요 버전"])

,기능,내 환경,필요 버전
0,"윈도우 함수 (RANK, LAG …)",OK,3.25+
1,NULLS FIRST / LAST,OK,3.30+
2,FILTER (WHERE …),OK,3.30+
3,IIF(),OK,3.32+
4,RIGHT / FULL JOIN,OK,3.39+
5,JSON 함수 json_extract,OK,3.38+ 기본 포함
6,JSON ->> 연산자,OK,3.38+
7,CONCAT(),OK,3.44+
8,STRING_AGG(),OK,3.44+
9,수학 함수 floor/sqrt/power,미지원,빌드에 따라 다름


## 3. 예제 테이블 `t`
아래 함수표의 집계·윈도우 예시에서 쓰는 작은 테이블입니다. (`amt`가 NULL인 행이 하나 들어 있습니다.)

In [3]:
sql("""
DROP TABLE IF EXISTS t;
CREATE TABLE t (n INTEGER, name TEXT, d TEXT, amt INTEGER);
INSERT INTO t VALUES
 (1, 'Max',     '2023-09-02', 125000),
 (2, ' Luna ',  '2023-09-15', 230000),
 (3, 'Coco',    '2023-10-01', NULL),
 (4, 'Bella',   '2023-10-12', 410000),
 (5, 'Max',     '2023-11-02', 142000);
SELECT * FROM t;
""")

,n,name,d,amt
0,1,Max,2023-09-02,125000.0
1,2,Luna,2023-09-15,230000.0
2,3,Coco,2023-10-01,NaN
3,4,Bella,2023-10-12,410000.0
4,5,Max,2023-11-02,142000.0


## 4. 함수표
각 표 아래의 코드 셀을 실행하면 같은 예시가 그대로 돌아갑니다.

## 날짜·시간 (날짜는 'YYYY-MM-DD' 텍스트)

| 구문 | 설명 | 예시 | 결과 |
|---|---|---|---|
| `date(값)` | 날짜 부분만 | `SELECT date('2021-04-11 13:45:00')` | 2021-04-11 |
| `strftime(형식, 값)` | 원하는 형식으로 출력 | `SELECT strftime('%Y/%m/%d', '2021-04-11')` | 2021/04/11 |
| `연도 숫자` | 연도를 정수로 | `SELECT CAST(strftime('%Y', '2021-04-11') AS INTEGER)` | 2021 |
| `월 숫자` | 월을 정수로 | `SELECT CAST(strftime('%m', '2021-04-11') AS INTEGER)` | 4 |
| `일 숫자` | 일을 정수로 | `SELECT CAST(strftime('%d', '2021-04-11') AS INTEGER)` | 11 |
| `요일` | 0=일 … 6=토 (텍스트) | `SELECT strftime('%w', '2021-04-11')` | 0 |
| `분기` | 월을 분기로 | `SELECT (CAST(strftime('%m', '2021-04-11') AS INTEGER) + 2) / 3` | 2 |
| `월 첫날` | 월 단위로 자르기 | `SELECT date('2021-04-11', 'start of month')` | 2021-04-01 |
| `월 마지막 날` | 다음 달 1일에서 하루 뺌 | `SELECT date('2021-04-11', 'start of month', '+1 month', '-1 day')` | 2021-04-30 |
| `일 더하기·빼기` | '+7 days', '-1 day' | `SELECT date('2021-04-11', '+7 days')` | 2021-04-18 |
| `월 더하기` | '+1 month' | `SELECT date('2021-04-11', '+1 month')` | 2021-05-11 |
| `일수 차이` | julianday 차이 | `SELECT CAST(julianday('2024-03-01') - julianday('2023-12-01') AS INTEGER)` | 91 |
| `월수 차이` | 연·월을 풀어서 계산 | `SELECT (strftime('%Y','2024-03-01') - strftime('%Y','2023-11-15')) * 12 + strftime('%m','2024-03-01') - strftime('%m','2023-11-15')` | 4 |

In [4]:
EXAMPLES = [
 [
  "date(값)",
  "SELECT date('2021-04-11 13:45:00')"
 ],
 [
  "strftime(형식, 값)",
  "SELECT strftime('%Y/%m/%d', '2021-04-11')"
 ],
 [
  "연도 숫자",
  "SELECT CAST(strftime('%Y', '2021-04-11') AS INTEGER)"
 ],
 [
  "월 숫자",
  "SELECT CAST(strftime('%m', '2021-04-11') AS INTEGER)"
 ],
 [
  "일 숫자",
  "SELECT CAST(strftime('%d', '2021-04-11') AS INTEGER)"
 ],
 [
  "요일",
  "SELECT strftime('%w', '2021-04-11')"
 ],
 [
  "분기",
  "SELECT (CAST(strftime('%m', '2021-04-11') AS INTEGER) + 2) / 3"
 ],
 [
  "월 첫날",
  "SELECT date('2021-04-11', 'start of month')"
 ],
 [
  "월 마지막 날",
  "SELECT date('2021-04-11', 'start of month', '+1 month', '-1 day')"
 ],
 [
  "일 더하기·빼기",
  "SELECT date('2021-04-11', '+7 days')"
 ],
 [
  "월 더하기",
  "SELECT date('2021-04-11', '+1 month')"
 ],
 [
  "일수 차이",
  "SELECT CAST(julianday('2024-03-01') - julianday('2023-12-01') AS INTEGER)"
 ],
 [
  "월수 차이",
  "SELECT (strftime('%Y','2024-03-01') - strftime('%Y','2023-11-15')) * 12 + strftime('%m','2024-03-01') - strftime('%m','2023-11-15')"
 ]
]

out = []
for label, q in EXAMPLES:
    try:
        res = con.execute(q).fetchall()
        out.append((label, res))
    except sqlite3.Error as e:
        out.append((label, "에러: " + str(e)))
pd.DataFrame(out, columns=["구문", "결과"])

,구문,결과
0,date(값),"[(2021-04-11,)]"
1,"strftime(형식, 값)","[(2021/04/11,)]"
2,연도 숫자,"[(2021,)]"
3,월 숫자,"[(4,)]"
4,일 숫자,"[(11,)]"
5,요일,"[(0,)]"
6,분기,"[(2,)]"
7,월 첫날,"[(2021-04-01,)]"
8,월 마지막 날,"[(2021-04-30,)]"
9,일 더하기·빼기,"[(2021-04-18,)]"


## 문자열

| 구문 | 설명 | 예시 | 결과 |
|---|---|---|---|
| `a \|\| b` | 이어 붙이기 (NULL이 끼면 NULL) | `SELECT 'a' \|\| 'b'` | ab |
| `CONCAT(a, b, …)` | 이어 붙이기, NULL은 빈 문자열 취급 (3.44+) | `SELECT CONCAT('a', NULL, 'b')` | ab |
| `UPPER / LOWER` | 대문자 / 소문자 | `SELECT UPPER('Max'), LOWER('Max')` | MAX, max |
| `LENGTH(s)` | 글자 수 (한글도 1글자) | `SELECT LENGTH('가나다')` | 3 |
| `TRIM / LTRIM / RTRIM` | 공백 제거 | `SELECT '[' \|\| TRIM('  Max  ') \|\| ']'` | [Max] |
| `SUBSTR(s, 시작, 길이)` | 부분 문자열 (1부터 시작) | `SELECT SUBSTR('hello', 2, 3)` | ell |
| `REPLACE(s, 찾기, 바꿀값)` | 문자열 치환 | `SELECT REPLACE('a-b-c', '-', '/')` | a/b/c |
| `INSTR(s, 찾기)` | 위치 (없으면 0) | `SELECT INSTR('hello', 'll')` | 3 |
| `LIKE` | 패턴 (영문은 대소문자 무시) | `SELECT 'Max' LIKE 'm%'` | 1 |
| `GLOB` | 패턴 (대소문자 구분, *와 ?) | `SELECT 'Max' GLOB 'm*'` | 0 |
| `printf(형식, 값)` | 소수 자리 고정 등 | `SELECT printf('%.2f', 50)` | 50.00 |

In [5]:
EXAMPLES = [
 [
  "a || b",
  "SELECT 'a' || 'b'"
 ],
 [
  "CONCAT(a, b, …)",
  "SELECT CONCAT('a', NULL, 'b')"
 ],
 [
  "UPPER / LOWER",
  "SELECT UPPER('Max'), LOWER('Max')"
 ],
 [
  "LENGTH(s)",
  "SELECT LENGTH('가나다')"
 ],
 [
  "TRIM / LTRIM / RTRIM",
  "SELECT '[' || TRIM('  Max  ') || ']'"
 ],
 [
  "SUBSTR(s, 시작, 길이)",
  "SELECT SUBSTR('hello', 2, 3)"
 ],
 [
  "REPLACE(s, 찾기, 바꿀값)",
  "SELECT REPLACE('a-b-c', '-', '/')"
 ],
 [
  "INSTR(s, 찾기)",
  "SELECT INSTR('hello', 'll')"
 ],
 [
  "LIKE",
  "SELECT 'Max' LIKE 'm%'"
 ],
 [
  "GLOB",
  "SELECT 'Max' GLOB 'm*'"
 ],
 [
  "printf(형식, 값)",
  "SELECT printf('%.2f', 50)"
 ]
]

out = []
for label, q in EXAMPLES:
    try:
        res = con.execute(q).fetchall()
        out.append((label, res))
    except sqlite3.Error as e:
        out.append((label, "에러: " + str(e)))
pd.DataFrame(out, columns=["구문", "결과"])

,구문,결과
0,a || b,"[(ab,)]"
1,"CONCAT(a, b, …)","[(ab,)]"
2,UPPER / LOWER,"[(MAX, max)]"
3,LENGTH(s),"[(3,)]"
4,TRIM / LTRIM / RTRIM,"[([Max],)]"
5,"SUBSTR(s, 시작, 길이)","[(ell,)]"
6,"REPLACE(s, 찾기, 바꿀값)","[(a/b/c,)]"
7,"INSTR(s, 찾기)","[(3,)]"
8,LIKE,"[(1,)]"
9,GLOB,"[(0,)]"


## 숫자·변환

| 구문 | 설명 | 예시 | 결과 |
|---|---|---|---|
| `ROUND(x, n)` | n자리까지 반올림 | `SELECT ROUND(2.567, 1)` | 2.6 |
| `ABS(x)` | 절댓값 | `SELECT ABS(-3)` | 3 |
| `MAX(a, b, …) / MIN(a, b, …)` | 인자가 여러 개면 집계가 아니라 행 안의 비교 | `SELECT MAX(1, 5, 3), MIN(1, 5, 3)` | 5, 1 |
| `CAST(x AS 타입)` | 형 변환 (INTEGER, REAL, TEXT) | `SELECT CAST('12' AS INTEGER) + 1` | 13 |
| `정수 ÷ 정수` | 소수점이 버려집니다 | `SELECT 7 / 2` | 3 |
| `x * 1.0 / y` | 소수 나눗셈으로 만들기 | `SELECT 7 * 1.0 / 2` | 3.5 |
| `typeof(x)` | 값의 저장 타입 확인 | `SELECT typeof(3.0), typeof('a'), typeof(NULL)` | real, text, null |
| `floor / ceil / sqrt / power` | 수학 함수 (빌드에 따라 없을 수 있음) | `SELECT floor(2.7), ceil(2.1), sqrt(16), power(2, 3)` | 2.0, 3.0, 4.0, 8.0 |

In [6]:
EXAMPLES = [
 [
  "ROUND(x, n)",
  "SELECT ROUND(2.567, 1)"
 ],
 [
  "ABS(x)",
  "SELECT ABS(-3)"
 ],
 [
  "MAX(a, b, …) / MIN(a, b, …)",
  "SELECT MAX(1, 5, 3), MIN(1, 5, 3)"
 ],
 [
  "CAST(x AS 타입)",
  "SELECT CAST('12' AS INTEGER) + 1"
 ],
 [
  "정수 ÷ 정수",
  "SELECT 7 / 2"
 ],
 [
  "x * 1.0 / y",
  "SELECT 7 * 1.0 / 2"
 ],
 [
  "typeof(x)",
  "SELECT typeof(3.0), typeof('a'), typeof(NULL)"
 ],
 [
  "floor / ceil / sqrt / power",
  "SELECT floor(2.7), ceil(2.1), sqrt(16), power(2, 3)"
 ]
]

out = []
for label, q in EXAMPLES:
    try:
        res = con.execute(q).fetchall()
        out.append((label, res))
    except sqlite3.Error as e:
        out.append((label, "에러: " + str(e)))
pd.DataFrame(out, columns=["구문", "결과"])

,구문,결과
0,"ROUND(x, n)","[(2.6,)]"
1,ABS(x),"[(3,)]"
2,"MAX(a, b, …) / MIN(a, b, …)","[(5, 1)]"
3,CAST(x AS 타입),"[(13,)]"
4,정수 ÷ 정수,"[(3,)]"
5,x * 1.0 / y,"[(3.5,)]"
6,typeof(x),"[(real, text, null)]"
7,floor / ceil / sqrt / power,에러: no such function: floor


## NULL·조건

| 구문 | 설명 | 예시 | 결과 |
|---|---|---|---|
| `COALESCE(a, b, …)` | 처음으로 NULL이 아닌 값 | `SELECT COALESCE(NULL, NULL, 7)` | 7 |
| `IFNULL(a, b)` | a가 NULL이면 b | `SELECT IFNULL(NULL, 'x')` | x |
| `NULLIF(a, b)` | 같으면 NULL (0으로 나누기 방지에 사용) | `SELECT NULLIF(0, 0), 10 * 1.0 / NULLIF(0, 0)` | NULL, NULL |
| `CASE WHEN … THEN … ELSE … END` | 조건 분기 | `SELECT CASE WHEN 5 >= 3 THEN 'high' ELSE 'low' END` | high |
| `IIF(조건, 참값, 거짓값)` | 짧은 분기 (3.32+) | `SELECT IIF(5 > 3, 'yes', 'no')` | yes |
| `IS NULL / IS NOT NULL` | NULL 검사 (= NULL은 쓰면 안 됨) | `SELECT NULL IS NULL, 3 IS NOT NULL` | 1, 1 |
| `IN (…) / BETWEEN a AND b` | 목록·범위 조건 | `SELECT 3 IN (1, 2, 3), 5 BETWEEN 1 AND 10` | 1, 1 |

In [7]:
EXAMPLES = [
 [
  "COALESCE(a, b, …)",
  "SELECT COALESCE(NULL, NULL, 7)"
 ],
 [
  "IFNULL(a, b)",
  "SELECT IFNULL(NULL, 'x')"
 ],
 [
  "NULLIF(a, b)",
  "SELECT NULLIF(0, 0), 10 * 1.0 / NULLIF(0, 0)"
 ],
 [
  "CASE WHEN … THEN … ELSE … END",
  "SELECT CASE WHEN 5 >= 3 THEN 'high' ELSE 'low' END"
 ],
 [
  "IIF(조건, 참값, 거짓값)",
  "SELECT IIF(5 > 3, 'yes', 'no')"
 ],
 [
  "IS NULL / IS NOT NULL",
  "SELECT NULL IS NULL, 3 IS NOT NULL"
 ],
 [
  "IN (…) / BETWEEN a AND b",
  "SELECT 3 IN (1, 2, 3), 5 BETWEEN 1 AND 10"
 ]
]

out = []
for label, q in EXAMPLES:
    try:
        res = con.execute(q).fetchall()
        out.append((label, res))
    except sqlite3.Error as e:
        out.append((label, "에러: " + str(e)))
pd.DataFrame(out, columns=["구문", "결과"])

,구문,결과
0,"COALESCE(a, b, …)","[(7,)]"
1,"IFNULL(a, b)","[(x,)]"
2,"NULLIF(a, b)","[(None, None)]"
3,CASE WHEN … THEN … ELSE … END,"[(high,)]"
4,"IIF(조건, 참값, 거짓값)","[(yes,)]"
5,IS NULL / IS NOT NULL,"[(1, 1)]"
6,IN (…) / BETWEEN a AND b,"[(1, 1)]"


## 집계 (예제 테이블 t 사용)

| 구문 | 설명 | 예시 | 결과 |
|---|---|---|---|
| `COUNT(*)` | 행 수 | `SELECT COUNT(*) FROM t` | 5 |
| `COUNT(열)` | NULL 제외 행 수 | `SELECT COUNT(amt) FROM t` | 4 |
| `COUNT(DISTINCT 열)` | 서로 다른 값의 수 | `SELECT COUNT(DISTINCT name) FROM t` | 4 |
| `SUM / AVG` | 합계 / 평균 (NULL 제외) | `SELECT SUM(amt), AVG(amt) FROM t` | 907000, 226750.0 |
| `MIN / MAX` | 최솟값 / 최댓값 | `SELECT MIN(amt), MAX(amt) FROM t` | 125000, 410000 |
| `TOTAL(열)` | SUM과 같지만 항상 실수, 모두 NULL이면 0.0 | `SELECT TOTAL(amt) FROM t` | 907000.0 |
| `GROUP_CONCAT(열, 구분자)` | 값을 한 문자열로 이음 (STRING_AGG도 3.44+) | `SELECT GROUP_CONCAT(TRIM(name), ',') FROM t` | Max,Luna,Coco,Bella,Max |
| `COUNT(*) FILTER (WHERE …)` | 조건에 맞는 행만 집계 | `SELECT COUNT(*) FILTER (WHERE amt > 200000) FROM t` | 2 |

In [8]:
EXAMPLES = [
 [
  "COUNT(*)",
  "SELECT COUNT(*) FROM t"
 ],
 [
  "COUNT(열)",
  "SELECT COUNT(amt) FROM t"
 ],
 [
  "COUNT(DISTINCT 열)",
  "SELECT COUNT(DISTINCT name) FROM t"
 ],
 [
  "SUM / AVG",
  "SELECT SUM(amt), AVG(amt) FROM t"
 ],
 [
  "MIN / MAX",
  "SELECT MIN(amt), MAX(amt) FROM t"
 ],
 [
  "TOTAL(열)",
  "SELECT TOTAL(amt) FROM t"
 ],
 [
  "GROUP_CONCAT(열, 구분자)",
  "SELECT GROUP_CONCAT(TRIM(name), ',') FROM t"
 ],
 [
  "COUNT(*) FILTER (WHERE …)",
  "SELECT COUNT(*) FILTER (WHERE amt > 200000) FROM t"
 ]
]

out = []
for label, q in EXAMPLES:
    try:
        res = con.execute(q).fetchall()
        out.append((label, res))
    except sqlite3.Error as e:
        out.append((label, "에러: " + str(e)))
pd.DataFrame(out, columns=["구문", "결과"])

,구문,결과
0,COUNT(*),"[(5,)]"
1,COUNT(열),"[(4,)]"
2,COUNT(DISTINCT 열),"[(4,)]"
3,SUM / AVG,"[(907000, 226750.0)]"
4,MIN / MAX,"[(125000, 410000)]"
5,TOTAL(열),"[(907000.0,)]"
6,"GROUP_CONCAT(열, 구분자)","[(Max,Luna,Coco,Bella,Max,)]"
7,COUNT(*) FILTER (WHERE …),"[(2,)]"


## 윈도우 함수 (예제 테이블 t 사용, 3.25+)

| 구문 | 설명 | 예시 | 결과 |
|---|---|---|---|
| `ROW_NUMBER()` | 행 번호 | `SELECT n, ROW_NUMBER() OVER (ORDER BY amt DESC) FROM t` | 4, 1 / 2, 2 / 5, 3 / 1, 4 / 3, 5 |
| `RANK()` | 동점이면 건너뜀 | `SELECT name, RANK() OVER (ORDER BY name) FROM t` |  Luna , 1 / Bella, 2 / Coco, 3 / Max, 4 / Max, 4 |
| `DENSE_RANK()` | 동점이어도 안 건너뜀 | `SELECT name, DENSE_RANK() OVER (ORDER BY name) FROM t` |  Luna , 1 / Bella, 2 / Coco, 3 / Max, 4 / Max, 4 |
| `NTILE(k)` | k등분 | `SELECT n, NTILE(2) OVER (ORDER BY n) FROM t` | 1, 1 / 2, 1 / 3, 1 / 4, 2 / 5, 2 |
| `LAG(열) / LEAD(열)` | 이전 / 다음 행 값 | `SELECT n, LAG(amt) OVER (ORDER BY n), LEAD(amt) OVER (ORDER BY n) FROM t` | 1, NULL, 230000 / 2, 125000, NULL / 3, 230000, 410000 / 4, NULL, 142000 / 5, 410000, NULL |
| `FIRST_VALUE(열)` | 정렬 기준 첫 값 | `SELECT n, FIRST_VALUE(amt) OVER (ORDER BY n) FROM t` | 1, 125000 / 2, 125000 / 3, 125000 / 4, 125000 / 5, 125000 |
| `SUM(열) OVER (ORDER BY …)` | 누적 합계 | `SELECT n, SUM(amt) OVER (ORDER BY n) FROM t` | 1, 125000 / 2, 355000 / 3, 355000 / 4, 765000 / 5, 907000 |
| `AVG(열) OVER (… ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` | 3행 이동평균 | `SELECT n, AVG(amt) OVER (ORDER BY n ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) FROM t` | 1, 125000.0 / 2, 177500.0 / 3, 177500.0 / 4, 320000.0 / 5, 276000.0 |
| `MAX(열) OVER (PARTITION BY …)` | 그룹별 최댓값을 각 행에 | `SELECT name, amt, MAX(amt) OVER (PARTITION BY name) FROM t` |  Luna , 230000, 230000 / Bella, 410000, 410000 / Coco, NULL, NULL / Max, 125000, 142000 / Max, 142000, 142000 |

In [9]:
EXAMPLES = [
 [
  "ROW_NUMBER()",
  "SELECT n, ROW_NUMBER() OVER (ORDER BY amt DESC) FROM t"
 ],
 [
  "RANK()",
  "SELECT name, RANK() OVER (ORDER BY name) FROM t"
 ],
 [
  "DENSE_RANK()",
  "SELECT name, DENSE_RANK() OVER (ORDER BY name) FROM t"
 ],
 [
  "NTILE(k)",
  "SELECT n, NTILE(2) OVER (ORDER BY n) FROM t"
 ],
 [
  "LAG(열) / LEAD(열)",
  "SELECT n, LAG(amt) OVER (ORDER BY n), LEAD(amt) OVER (ORDER BY n) FROM t"
 ],
 [
  "FIRST_VALUE(열)",
  "SELECT n, FIRST_VALUE(amt) OVER (ORDER BY n) FROM t"
 ],
 [
  "SUM(열) OVER (ORDER BY …)",
  "SELECT n, SUM(amt) OVER (ORDER BY n) FROM t"
 ],
 [
  "AVG(열) OVER (… ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)",
  "SELECT n, AVG(amt) OVER (ORDER BY n ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) FROM t"
 ],
 [
  "MAX(열) OVER (PARTITION BY …)",
  "SELECT name, amt, MAX(amt) OVER (PARTITION BY name) FROM t"
 ]
]

out = []
for label, q in EXAMPLES:
    try:
        res = con.execute(q).fetchall()
        out.append((label, res))
    except sqlite3.Error as e:
        out.append((label, "에러: " + str(e)))
pd.DataFrame(out, columns=["구문", "결과"])

,구문,결과
0,ROW_NUMBER(),"[(4, 1), (2, 2), (5, 3), (1, 4), (3, 5)]"
1,RANK(),"[( Luna , 1), (Bella, 2), (Coco, 3), (Max, 4),..."
2,DENSE_RANK(),"[( Luna , 1), (Bella, 2), (Coco, 3), (Max, 4),..."
3,NTILE(k),"[(1, 1), (2, 1), (3, 1), (4, 2), (5, 2)]"
4,LAG(열) / LEAD(열),"[(1, None, 230000), (2, 125000, None), (3, 230..."
5,FIRST_VALUE(열),"[(1, 125000), (2, 125000), (3, 125000), (4, 12..."
6,SUM(열) OVER (ORDER BY …),"[(1, 125000), (2, 355000), (3, 355000), (4, 76..."
7,AVG(열) OVER (… ROWS BETWEEN 2 PRECEDING AND CU...,"[(1, 125000.0), (2, 177500.0), (3, 177500.0), ..."
8,MAX(열) OVER (PARTITION BY …),"[( Luna , 230000, 230000), (Bella, 410000, 410..."


## JSON·구조 확인

| 구문 | 설명 | 예시 | 결과 |
|---|---|---|---|
| `json_extract(json, '$.키')` | JSON에서 값 꺼내기 | `SELECT json_extract('{"user": "john", "n": 3}', '$.user')` | john |
| `json ->> '$.키'` | 같은 동작의 연산자 (3.38+) | `SELECT '{"user": "john"}' ->> '$.user'` | john |
| `PRAGMA table_info(테이블)` | 열 이름·타입 확인 (DESCRIBE 대신) | `SELECT name, type FROM pragma_table_info('t')` | n, INTEGER / name, TEXT / d, TEXT / amt, INTEGER |
| `sqlite_master` | 만들어진 테이블 목록 | `SELECT name FROM sqlite_master WHERE type = 'table'` | t |

In [10]:
EXAMPLES = [
 [
  "json_extract(json, '$.키')",
  "SELECT json_extract('{\"user\": \"john\", \"n\": 3}', '$.user')"
 ],
 [
  "json ->> '$.키'",
  "SELECT '{\"user\": \"john\"}' ->> '$.user'"
 ],
 [
  "PRAGMA table_info(테이블)",
  "SELECT name, type FROM pragma_table_info('t')"
 ],
 [
  "sqlite_master",
  "SELECT name FROM sqlite_master WHERE type = 'table'"
 ]
]

out = []
for label, q in EXAMPLES:
    try:
        res = con.execute(q).fetchall()
        out.append((label, res))
    except sqlite3.Error as e:
        out.append((label, "에러: " + str(e)))
pd.DataFrame(out, columns=["구문", "결과"])

,구문,결과
0,"json_extract(json, '$.키')","[(john,)]"
1,json ->> '$.키',"[(john,)]"
2,PRAGMA table_info(테이블),"[(n, INTEGER), (name, TEXT), (d, TEXT), (amt, ..."
3,sqlite_master,"[(t,)]"


## 5. SQLite에 없는 구문과 대체 방법
다른 DB(MySQL·BigQuery·DuckDB) 문법을 그대로 쓰면 에러가 나는 것들입니다. 아래는 모두 SQLite 3.45.1에서 실제로 에러가 나는 것을 확인했습니다.

| 구문 | SQLite에서는 |
|---|---|
| `ROLLUP / CUBE / GROUPING()` | `UNION ALL`로 상세·소계·총계 쿼리를 직접 이어 붙임 |
| `GROUPING SETS` | `UNION ALL`로 직접 구현 |
| `CREATE OR REPLACE TABLE` | `DROP TABLE IF EXISTS z;` 후 `CREATE TABLE z(...)` |
| `UNION DISTINCT` | `UNION` (기본이 중복 제거) |
| `DATE_TRUNC(…)` | `date(d, 'start of month')` |
| `EXTRACT(YEAR FROM d)` | `CAST(strftime('%Y', d) AS INTEGER)` |
| `YEAR(d) / MONTH(d) / DAY(d)` | `CAST(strftime('%Y', d) AS INTEGER)` 등 |
| `DATE_FORMAT(d, 형식)` | `strftime('%Y', d)` (인자 순서가 반대) |
| `DATE_DIFF(…)` | `julianday(끝) - julianday(시작)` |
| `COUNTIF(조건)` | `SUM(CASE WHEN 조건 THEN 1 ELSE 0 END)` 또는 `COUNT(*) FILTER (WHERE 조건)` |
| `ILIKE` | `LIKE` (영문은 원래 대소문자 무시) |
| `DESCRIBE 테이블` | `PRAGMA table_info(t);` |
| `마지막 열 뒤 쉼표` | 마지막 열 뒤의 쉼표를 지움 |

## 6. 알아 둘 SQLite 특징
- **정수 ÷ 정수는 소수점이 버려집니다.** `7 / 2`는 3입니다. 비율·평균을 구할 때는 `x * 1.0 / y`로 쓰세요.
- **날짜 타입이 없습니다.** `'YYYY-MM-DD'` 텍스트로 저장하고, 이 형식일 때만 `date()`·`strftime()`이 제대로 동작합니다.
- **열 타입 선언은 강제되지 않습니다.** `INTEGER` 열에 문자열을 넣어도 들어갑니다(숫자처럼 보이면 변환됨). `STRING`, `INT64`, `DECIMAL`은 SQLite 타입 이름이 아니라서 `TEXT`, `INTEGER`, `REAL`로 씁니다.
- **PRIMARY KEY, FOREIGN KEY, UNIQUE**: 선언은 되지만 외래 키는 `PRAGMA foreign_keys = ON;`을 켜야 검사합니다.
- **`UPDATE`/`DELETE`는 `WHERE`가 없어도 실행됩니다.** 전체 행이 바뀌므로 먼저 같은 조건으로 `SELECT`해서 확인하세요.
- **`ORDER BY` 없이는 행 순서가 보장되지 않습니다.** 결과를 비교하려면 정렬 기준을 꼭 붙이세요.